# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedant08mehta/Flyrank-assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/vedant08mehta/Flyrank-assignment1.git
%cd /content/Flyrank-assignment1

Cloning into 'Flyrank-assignment1'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 161 (delta 67), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (161/161), 1.90 MiB | 17.52 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/Flyrank-assignment1


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: The research suggests that search-performance signals can be useful for identifying pages that may need attention. The label in our work comes from the observed trend direction rather than from a manually assigned judgment. This supports the claim as an observed relationship, but the validation design determines how confidently the result can generalize.

Finding 2: The research also emphasizes that content and search-performance characteristics can be associated with changes in organic performance. In our workflow, these relationships are evaluated using held-out data rather than assuming that an association proves causation. A stronger validation design therefore makes the finding more useful for decision-support.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Two findings reviewed.")

Two findings reviewed.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I will re-evaluate the Week-5 Random Forest using a client-grouped holdout. This is a more honest test because pages from the same client cannot appear in both training and test sets. I will compare the resulting Precision@50 with the earlier result to see whether the model's performance holds when evaluated on unseen clients.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update"
]

model_data = df.dropna(
    subset=feature_cols + [
        "is_declining_label",
        "client_id"
    ]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining_label"]
groups = model_data["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print(f"Train rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Train clients: {groups_train.nunique():,}")
print(f"Test clients: {groups_test.nunique():,}")

print(
    "Client overlap:",
    len(set(groups_train) & set(groups_test))
)

Train rows: 22,885
Test rows: 7,115
Train clients: 24
Test clients: 8
Client overlap: 0


In [5]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(
    X_train,
    y_train
)

test_proba = model.predict_proba(
    X_test
)[:, 1]

top50_idx = np.argsort(
    test_proba
)[::-1][:50]

precision_50 = y_test.iloc[
    top50_idx
].mean()

print(
    f"Honest-split Precision@50: "
    f"{precision_50:.3f}"
)

print(
    f"Test-set base rate: "
    f"{y_test.mean():.3f}"
)

Honest-split Precision@50: 0.680
Test-set base rate: 0.517


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature set must contain information available before the prediction is made. I will specifically check that the target-derived fields trend_direction, trend_pct, and is_declining_label are not included as model features. I will also check that the selected features do not directly encode the outcome.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("=== Leakage audit ===")

for field in leakage_fields:
    print(
        f"{field}: "
        f"{'USED' if field in feature_cols else 'NOT USED'}"
    )

assert not any(
    field in feature_cols
    for field in leakage_fields
)

print(
    "\nPASS: no target-derived fields "
    "are included in the feature set."
)

=== Leakage audit ===
trend_direction: NOT USED
trend_pct: NOT USED
is_declining_label: NOT USED

PASS: no target-derived fields are included in the feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The Week-5 model showed higher Precision@50 than the hand-written baseline on the evaluated test clients, indicating that the learned model captured useful patterns beyond the simple rule. Under the client-grouped validation, the model achieved a Precision@50 of 0.680, which was above the test-set base rate of 0.517. This should be described as an observed and directional result rather than proof that the model will perform identically on every future client. The model is best treated as decision-support for prioritizing pages for review, not as causal proof or a prediction of Google's algorithm.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(
    "Observed honest-split Precision@50: 0.680"
)

print(
    "Observed test-set base rate: 0.517"
)

print(
    "Interpretation: the model provided "
    "useful directional decision-support "
    "on the held-out clients."
)

print(
    "Limitation: this does not establish "
    "causation or predict Google's algorithm."
)

Observed honest-split Precision@50: 0.680
Observed test-set base rate: 0.517
Interpretation: the model provided useful directional decision-support on the held-out clients.
Limitation: this does not establish causation or predict Google's algorithm.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.